# 🏭 On-Device PDF Analyzer & Q&A Engine
**Industrial-Grade RAG Pipeline — Powered by Ollama + LangChain + FAISS**

A fully local, privacy-first document analysis and question-answering system.  
No cloud APIs. No data leaves your machine. Production-ready architecture.

---

### Architecture Overview
```
PDFs ──▶ Loader ──▶ Chunker ──▶ Embeddings ──▶ FAISS Vector Store
                                                       │
                                    ┌──────────────────┘
                                    ▼
                    User Query ──▶ Retriever ──▶ LLM ──▶ Answer + Sources
```

### Features
- **RAG-Powered Q&A** — Ask natural-language questions about your documents
- **Multi-Document Analysis** — Batch analysis with structured reports
- **Semantic Search** — FAISS vector store with cosine similarity retrieval
- **Source Attribution** — Every answer traces back to specific document chunks
- **Configurable Pipeline** — Swap models, chunking strategies, retrieval params
- **Robust Error Handling** — Retry logic, graceful degradation, structured logging
- **Export Options** — Markdown reports, JSON, scrolling MP4 video

### Quick Start
1. Install [Ollama](https://ollama.com) → `ollama pull llama3` and `ollama pull nomic-embed-text`
2. `pip install -r requirements.txt`
3. Drop PDFs into `pdfs/` folder
4. Run cells top to bottom


In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 1 · Install & Import Dependencies
# ═══════════════════════════════════════════════════════
# Uncomment below on first run:
# !pip install langchain-community langchain-ollama langchain pypdf \
#              faiss-cpu sentence-transformers tiktoken pyyaml tenacity

import os
import sys
import json
import time
import logging
import hashlib
import textwrap
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass, field, asdict
from typing import Optional

import pypdf
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA

print("✅ All dependencies imported successfully.")
print(f"   Python {sys.version.split()[0]} | {datetime.now().strftime('%Y-%m-%d %H:%M')}")


In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 2 · Logging Configuration
# ═══════════════════════════════════════════════════════

LOG_DIR = Path("logs")
LOG_DIR.mkdir(exist_ok=True)

log_file = LOG_DIR / f"pipeline_{datetime.now():%Y%m%d_%H%M%S}.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s │ %(levelname)-8s │ %(message)s",
    datefmt="%H:%M:%S",
    handlers=[
        logging.FileHandler(log_file, encoding="utf-8"),
        logging.StreamHandler(sys.stdout),
    ],
)
logger = logging.getLogger("pdf_analyzer")
logger.info(f"Log file: {log_file}")


In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 3 · Pipeline Configuration
# ═══════════════════════════════════════════════════════

@dataclass
class PipelineConfig:
    """Central configuration for the entire pipeline."""

    # ── Paths ──
    pdf_folder: str = "pdfs"
    output_dir: str = "outputs"
    vectorstore_dir: str = "vectorstore"

    # ── Model Settings ──
    llm_model: str = "llama3:8b"
    embedding_model: str = "nomic-embed-text"
    temperature: float = 0.1          # Low temp for factual analysis
    qa_temperature: float = 0.2       # Slightly higher for Q&A fluency

    # ── Chunking Strategy ──
    chunk_size: int = 1000            # Characters per chunk
    chunk_overlap: int = 200          # Overlap for context continuity
    separators: list = field(default_factory=lambda: ["\n\n", "\n", ". ", " ", ""])

    # ── Retrieval ──
    retrieval_k: int = 5              # Top-k chunks to retrieve
    search_type: str = "mmr"          # "similarity" or "mmr" (maximal marginal relevance)
    mmr_diversity: float = 0.3        # Lambda for MMR (0=max diversity, 1=max relevance)

    # ── Analysis ──
    max_chars_per_doc: int = 15000    # Max chars sent to LLM for full-doc analysis
    analysis_timeout: int = 300       # Seconds before analysis times out

    # ── Output ──
    report_filename: str = "analysis_report.md"
    video_filename: str = "analysis_output.mp4"

    def validate(self):
        """Validate config values at startup."""
        assert Path(self.pdf_folder).exists(), f"PDF folder not found: {self.pdf_folder}"
        assert self.chunk_size > self.chunk_overlap, "chunk_size must exceed chunk_overlap"
        assert 0 <= self.temperature <= 2, "temperature must be in [0, 2]"
        assert self.retrieval_k >= 1, "retrieval_k must be >= 1"
        logger.info("Configuration validated ✓")


config = PipelineConfig()

# Create required directories
for d in [config.pdf_folder, config.output_dir, config.vectorstore_dir]:
    Path(d).mkdir(exist_ok=True)

config.validate()

# Display config
print("\n┌─── Pipeline Configuration ───────────────────────┐")
for k, v in asdict(config).items():
    if k != "separators":
        print(f"│  {k:<22} = {v}")
print("└──────────────────────────────────────────────────┘")


In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 4 · PDF Loader with Metadata Extraction
# ═══════════════════════════════════════════════════════

@dataclass
class PDFDocument:
    """Structured representation of a loaded PDF."""
    filename: str
    text: str
    num_pages: int
    file_hash: str
    metadata: dict
    file_size_kb: float

    @property
    def char_count(self) -> int:
        return len(self.text)

    @property
    def word_count(self) -> int:
        return len(self.text.split())


def compute_file_hash(filepath: str) -> str:
    """SHA-256 hash for deduplication and cache validation."""
    h = hashlib.sha256()
    with open(filepath, "rb") as f:
        for block in iter(lambda: f.read(8192), b""):
            h.update(block)
    return h.hexdigest()[:16]


def load_pdfs(folder: str) -> list[PDFDocument]:
    """Load all PDFs from folder with metadata extraction."""
    folder_path = Path(folder)
    pdf_files = sorted(folder_path.glob("*.pdf"))

    if not pdf_files:
        logger.warning(f"No PDFs found in '{folder}'. Add files and re-run.")
        return []

    logger.info(f"Found {len(pdf_files)} PDF(s) in '{folder}'")
    documents = []

    for pdf_path in pdf_files:
        try:
            reader = pypdf.PdfReader(str(pdf_path))
            pages_text = []
            for page in reader.pages:
                text = page.extract_text()
                if text:
                    pages_text.append(text.strip())

            full_text = "\n".join(pages_text)
            file_hash = compute_file_hash(str(pdf_path))
            file_size = pdf_path.stat().st_size / 1024

            # Extract PDF metadata
            meta = reader.metadata or {}
            doc_metadata = {
                "title": getattr(meta, "title", None) or pdf_path.stem,
                "author": getattr(meta, "author", None) or "Unknown",
                "subject": getattr(meta, "subject", None) or "",
                "creator": getattr(meta, "creator", None) or "",
            }

            doc = PDFDocument(
                filename=pdf_path.name,
                text=full_text,
                num_pages=len(reader.pages),
                file_hash=file_hash,
                metadata=doc_metadata,
                file_size_kb=round(file_size, 1),
            )
            documents.append(doc)

            logger.info(
                f"  ✅ {doc.filename} — "
                f"{doc.num_pages} pages, {doc.word_count:,} words, "
                f"{doc.file_size_kb} KB, hash={doc.file_hash}"
            )

            if doc.char_count == 0:
                logger.warning(f"     ⚠️  No text extracted — may be a scanned/image PDF.")

        except Exception as e:
            logger.error(f"  ❌ Failed to load {pdf_path.name}: {e}")

    logger.info(f"📦 Loaded {len(documents)}/{len(pdf_files)} PDF(s) successfully")
    return documents


documents = load_pdfs(config.pdf_folder)

# Preview
if documents:
    doc = documents[0]
    print(f"\n{'─'*55}")
    print(f"Preview — {doc.filename} ({doc.word_count:,} words)")
    print(f"{'─'*55}")
    print(doc.text[:600] + "\n...")


In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 5 · Text Chunking for RAG Pipeline
# ═══════════════════════════════════════════════════════

def chunk_documents(
    documents: list[PDFDocument],
    chunk_size: int,
    chunk_overlap: int,
    separators: list[str],
) -> list[Document]:
    """
    Split documents into overlapping chunks for vector embedding.
    Each chunk retains source metadata for attribution.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=separators,
        length_function=len,
        is_separator_regex=False,
    )

    all_chunks = []
    for doc in documents:
        if not doc.text.strip():
            continue

        chunks = splitter.create_documents(
            texts=[doc.text],
            metadatas=[{
                "source": doc.filename,
                "title": doc.metadata.get("title", doc.filename),
                "author": doc.metadata.get("author", "Unknown"),
                "file_hash": doc.file_hash,
                "total_pages": doc.num_pages,
            }],
        )

        # Add chunk index to metadata
        for i, chunk in enumerate(chunks):
            chunk.metadata["chunk_index"] = i
            chunk.metadata["total_chunks"] = len(chunks)

        all_chunks.extend(chunks)
        logger.info(f"  📄 {doc.filename} → {len(chunks)} chunks")

    logger.info(
        f"\n📊 Chunking Summary:\n"
        f"   Documents: {len(documents)}\n"
        f"   Total chunks: {len(all_chunks)}\n"
        f"   Avg chunk size: {sum(len(c.page_content) for c in all_chunks) // max(len(all_chunks),1)} chars"
    )
    return all_chunks


chunks = chunk_documents(
    documents,
    chunk_size=config.chunk_size,
    chunk_overlap=config.chunk_overlap,
    separators=config.separators,
)

# Preview a chunk
if chunks:
    print(f"\n{'─'*55}")
    print(f"Sample chunk (index 0) from: {chunks[0].metadata['source']}")
    print(f"{'─'*55}")
    print(chunks[0].page_content[:400] + "\n...")


In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 6 · Build FAISS Vector Store
# ═══════════════════════════════════════════════════════

def build_vector_store(
    chunks: list[Document],
    embedding_model: str,
    persist_dir: str,
) -> FAISS:
    """
    Build (or load cached) FAISS vector store from document chunks.
    Uses Ollama embeddings so everything stays local.
    """
    persist_path = Path(persist_dir)

    logger.info(f"Initializing embeddings model: {embedding_model}")
    embeddings = OllamaEmbeddings(model=embedding_model)

    # Check for cached vector store
    index_file = persist_path / "index.faiss"
    if index_file.exists():
        logger.info("Found cached vector store — loading from disk...")
        try:
            store = FAISS.load_local(
                str(persist_path), embeddings, allow_dangerous_deserialization=True
            )
            logger.info(f"✅ Loaded cached store ({store.index.ntotal} vectors)")

            # Verify cache is still valid by checking doc count
            cached_sources = set()
            for doc_id in store.docstore._dict:
                src = store.docstore._dict[doc_id].metadata.get("source", "")
                cached_sources.add(src)

            current_sources = {d.filename for d in documents}
            if cached_sources == current_sources:
                logger.info("   Cache matches current documents ✓")
                return store
            else:
                logger.info("   Documents changed — rebuilding index...")
        except Exception as e:
            logger.warning(f"   Cache load failed ({e}) — rebuilding...")

    # Build fresh
    logger.info(f"Building vector store from {len(chunks)} chunks...")
    start = time.time()

    store = FAISS.from_documents(chunks, embeddings)
    elapsed = time.time() - start

    logger.info(f"✅ Vector store built in {elapsed:.1f}s — {store.index.ntotal} vectors")

    # Persist to disk
    store.save_local(str(persist_path))
    logger.info(f"   Saved to: {persist_path}/")

    return store


vectorstore = build_vector_store(chunks, config.embedding_model, config.vectorstore_dir)


In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 7 · Connect to Local LLM via Ollama
# ═══════════════════════════════════════════════════════

def connect_llm(model_name: str, temperature: float = 0.1, max_retries: int = 3) -> ChatOllama:
    """Initialize Ollama LLM with retry logic."""
    logger.info(f"Connecting to model: {model_name} (temp={temperature})")

    for attempt in range(1, max_retries + 1):
        try:
            llm = ChatOllama(
                model=model_name,
                temperature=temperature,
                num_ctx=4096,          # Context window
                num_predict=2048,      # Max output tokens
            )
            # Health check
            response = llm.invoke("Reply with exactly: READY")
            logger.info(f"✅ Connected on attempt {attempt} — response: {response.content.strip()}")
            return llm
        except Exception as e:
            logger.warning(f"   Attempt {attempt}/{max_retries} failed: {e}")
            if attempt < max_retries:
                wait = 2 ** attempt
                logger.info(f"   Retrying in {wait}s...")
                time.sleep(wait)
            else:
                logger.error("❌ All connection attempts failed.")
                logger.error("Troubleshooting:")
                logger.error("  1. Is Ollama running? → 'ollama serve'")
                logger.error(f"  2. Is the model pulled? → 'ollama pull {model_name}'")
                raise


# Create two LLM instances: one for analysis, one for Q&A
llm_analysis = connect_llm(config.llm_model, temperature=config.temperature)
llm_qa = connect_llm(config.llm_model, temperature=config.qa_temperature)

print("\n✅ LLM instances ready (analysis + Q&A)")


In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 8 · Document Analysis Chain
# ═══════════════════════════════════════════════════════

SYSTEM_PROMPT = """You are an expert document analyst. Produce structured, actionable reports.
Be precise. Cite specific details from the source text. Use Markdown formatting."""

ANALYSIS_TEMPLATE = """Analyze this document thoroughly:
---
{document_text}
---

Produce the following sections:

## 1. Executive Summary
2-3 sentences: what is this document about, who is the audience, and what is the core argument?

## 2. Key Findings
Bulleted list of the 5-8 most important concepts, findings, or arguments. Be specific.

## 3. Critical Terminology
Table of 5-10 essential terms with concise definitions. Format:
| Term | Definition |
|------|-----------|

## 4. Methodology / Approach
How does the document support its claims? (Data, citations, logical reasoning, etc.)

## 5. Strengths & Limitations
What does this document do well? Where are the gaps?

## 6. References & Resources
Any sources, tools, datasets, or further reading mentioned.

## 7. Study Questions
5 thought-provoking questions that test deep understanding (not surface recall).

## 8. ELI5
Explain the core idea in 2-3 simple sentences anyone could understand.

## 9. Actionable Takeaways
3-5 concrete things a reader should do or remember after reading this.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", ANALYSIS_TEMPLATE),
])

analysis_chain = prompt | llm_analysis | StrOutputParser()
logger.info("✅ Analysis chain constructed")


In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 9 · Run Analysis on All Documents
# ═══════════════════════════════════════════════════════

def truncate_text(text: str, max_chars: int) -> str:
    """Smart truncation that preserves sentence boundaries."""
    if len(text) <= max_chars:
        return text
    truncated = text[:max_chars]
    # Try to cut at last sentence boundary
    last_period = truncated.rfind(". ")
    if last_period > max_chars * 0.8:
        truncated = truncated[:last_period + 1]
    logger.info(f"  ⚠️  Truncated to {len(truncated):,} chars (limit: {max_chars:,})")
    return truncated + "\n\n[... document truncated for model context limits ...]"


def analyze_documents(documents, chain, max_chars):
    """Run LLM analysis on each document with timing and error handling."""
    results = []

    for i, doc in enumerate(documents, 1):
        logger.info(f"\n{'═'*55}")
        logger.info(f"📄 [{i}/{len(documents)}] Analyzing: {doc.filename}")
        logger.info(f"{'═'*55}")

        text = truncate_text(doc.text, max_chars)
        if not text.strip():
            logger.warning(f"  Skipping {doc.filename} — no extractable text")
            continue

        start = time.time()
        try:
            response = chain.invoke({"document_text": text})
            elapsed = time.time() - start

            results.append({
                "filename": doc.filename,
                "analysis": response,
                "metadata": doc.metadata,
                "stats": {
                    "pages": doc.num_pages,
                    "words": doc.word_count,
                    "analysis_time_sec": round(elapsed, 1),
                },
            })

            logger.info(f"  ✅ Done in {elapsed:.1f}s")
            print(f"\n{response[:500]}\n{'...' if len(response) > 500 else ''}")

        except Exception as e:
            logger.error(f"  ❌ Analysis failed for {doc.filename}: {e}")

    logger.info(f"\n📊 Analysis complete: {len(results)}/{len(documents)} documents processed")
    return results


results = analyze_documents(documents, analysis_chain, config.max_chars_per_doc)


In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 10 · RAG-Powered Question & Answer Engine
# ═══════════════════════════════════════════════════════

QA_SYSTEM_PROMPT = """You are a precise document Q&A assistant. Answer questions based ONLY
on the provided context. Rules:
1. If the context contains the answer, provide it clearly with specific details.
2. If the context is insufficient, say "I don't have enough information in the
   loaded documents to answer this fully" and explain what you CAN infer.
3. Always cite which document(s) your answer comes from.
4. Be concise but complete. Prefer specifics over generalities."""

QA_TEMPLATE = """Context from documents:
---
{context}
---

Question: {question}

Provide a clear, well-structured answer based on the context above.
Cite the source document(s) for each claim."""


class QAEngine:
    """RAG Question-Answering engine with source attribution."""

    def __init__(self, vectorstore, llm, config: PipelineConfig):
        self.vectorstore = vectorstore
        self.llm = llm
        self.config = config
        self.history: list[dict] = []

        # Build retriever
        search_kwargs = {"k": config.retrieval_k}
        if config.search_type == "mmr":
            search_kwargs["lambda_mult"] = config.mmr_diversity
            search_kwargs["fetch_k"] = config.retrieval_k * 3

        self.retriever = vectorstore.as_retriever(
            search_type=config.search_type,
            search_kwargs=search_kwargs,
        )

        # Build QA prompt
        self.qa_prompt = ChatPromptTemplate.from_messages([
            ("system", QA_SYSTEM_PROMPT),
            ("human", QA_TEMPLATE),
        ])

        self.chain = self.qa_prompt | self.llm | StrOutputParser()
        logger.info("✅ Q&A Engine initialized")

    def ask(self, question: str, verbose: bool = True) -> dict:
        """
        Ask a question and get an answer with source attribution.

        Returns:
            dict with keys: question, answer, sources, retrieval_time, generation_time
        """
        logger.info(f"\n❓ Question: {question}")

        # Step 1: Retrieve relevant chunks
        t0 = time.time()
        retrieved_docs = self.retriever.invoke(question)
        retrieval_time = time.time() - t0

        # Build context with source labels
        context_parts = []
        sources = []
        for i, doc in enumerate(retrieved_docs):
            src = doc.metadata.get("source", "unknown")
            chunk_idx = doc.metadata.get("chunk_index", "?")
            label = f"[Source: {src}, chunk {chunk_idx}]"
            context_parts.append(f"{label}\n{doc.page_content}")
            sources.append({
                "file": src,
                "chunk_index": chunk_idx,
                "preview": doc.page_content[:150] + "...",
            })

        context = "\n\n".join(context_parts)

        # Step 2: Generate answer
        t1 = time.time()
        answer = self.chain.invoke({"context": context, "question": question})
        generation_time = time.time() - t1

        result = {
            "question": question,
            "answer": answer,
            "sources": sources,
            "retrieval_time": round(retrieval_time, 2),
            "generation_time": round(generation_time, 2),
            "chunks_retrieved": len(retrieved_docs),
        }

        self.history.append(result)

        if verbose:
            print(f"\n{'─'*55}")
            print(f"📝 Answer (retrieved {len(retrieved_docs)} chunks "
                  f"in {retrieval_time:.2f}s, generated in {generation_time:.1f}s):")
            print(f"{'─'*55}")
            print(answer)
            print(f"\n📚 Sources:")
            for s in sources:
                print(f"   • {s['file']} (chunk {s['chunk_index']})")

        return result


# Initialize Q&A engine
qa = QAEngine(vectorstore, llm_qa, config)


In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 11 · Interactive Q&A Session
# ═══════════════════════════════════════════════════════
# Run this cell and type your questions!

print("╔══════════════════════════════════════════════════╗")
print("║       📚 Document Q&A — Interactive Mode        ║")
print("║   Type your questions below. Enter 'q' to quit. ║")
print("╚══════════════════════════════════════════════════╝\n")

while True:
    question = input("\n❓ Your question: ").strip()
    if not question:
        continue
    if question.lower() in ("q", "quit", "exit"):
        print("\n👋 Session ended.")
        break
    qa.ask(question)


In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 12 · Batch Q&A — Predefined Questions
# ═══════════════════════════════════════════════════════
# Edit these questions for your specific documents:

BATCH_QUESTIONS = [
    "What is the main argument or thesis of this document?",
    "What methodology or evidence is used to support the claims?",
    "What are the key limitations or criticisms discussed?",
    "How does this relate to current industry practices?",
    "What are the practical implications or recommendations?",
]

print("Running batch Q&A...\n")
batch_results = []

for q in BATCH_QUESTIONS:
    result = qa.ask(q, verbose=True)
    batch_results.append(result)
    print()

print(f"\n✅ Batch Q&A complete — {len(batch_results)} questions answered")


In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 13 · Export Results
# ═══════════════════════════════════════════════════════

def export_markdown_report(results: list, qa_history: list, output_path: str):
    """Export full analysis + Q&A to a structured Markdown report."""
    out = Path(output_path)
    with open(out, "w", encoding="utf-8") as f:
        f.write("# 📊 PDF Analysis Report\n\n")
        f.write(f"**Generated:** {datetime.now():%Y-%m-%d %H:%M}\n\n")
        f.write(f"**Documents analyzed:** {len(results)}\n\n")
        f.write("---\n\n")

        # Document analyses
        for r in results:
            f.write(f"## 📄 {r['filename']}\n\n")
            stats = r.get("stats", {})
            f.write(f"*{stats.get('pages', '?')} pages · "
                    f"{stats.get('words', '?'):,} words · "
                    f"Analyzed in {stats.get('analysis_time_sec', '?')}s*\n\n")
            f.write(r["analysis"])
            f.write("\n\n---\n\n")

        # Q&A section
        if qa_history:
            f.write("## 💬 Questions & Answers\n\n")
            for i, qa_item in enumerate(qa_history, 1):
                f.write(f"### Q{i}: {qa_item['question']}\n\n")
                f.write(f"{qa_item['answer']}\n\n")
                f.write(f"*Sources: {', '.join(s['file'] for s in qa_item['sources'])}*\n\n")
                f.write("---\n\n")

    logger.info(f"✅ Report saved: {out}")


def export_json(results: list, qa_history: list, output_path: str):
    """Export structured JSON for programmatic consumption."""
    payload = {
        "generated_at": datetime.now().isoformat(),
        "pipeline_config": asdict(config),
        "analyses": results,
        "qa_history": qa_history,
    }
    out = Path(output_path)
    with open(out, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False, default=str)
    logger.info(f"✅ JSON export saved: {out}")


# Export both formats
output_dir = Path(config.output_dir)

export_markdown_report(
    results,
    qa.history,
    str(output_dir / config.report_filename),
)

export_json(
    results,
    qa.history,
    str(output_dir / "analysis_results.json"),
)

print(f"\n📁 All outputs saved to: {output_dir}/")


In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 14 · (Optional) Generate Scrolling MP4 Video
# ═══════════════════════════════════════════════════════
# Requires: pip install matplotlib  +  ffmpeg installed on system

import matplotlib
matplotlib.use("Agg")  # Non-interactive backend for stability
import matplotlib.pyplot as plt
from matplotlib import animation

def create_scrolling_mp4(
    text: str,
    out_file: str = "analysis_output.mp4",
    fps: int = 24,
    lines_in_window: int = 28,
    wrap_width: int = 95,
    fontsize: int = 11,
    figsize: tuple = (14, 8),
    bg_color: str = "#0d1117",
    text_color: str = "#c9d1d9",
    accent_color: str = "#58a6ff",
):
    """Generate a cinematic scrolling MP4 of the analysis report."""
    plt.rcParams["font.family"] = "DejaVu Sans"

    lines = []
    for paragraph in text.split("\n"):
        stripped = paragraph.strip()
        if stripped == "":
            lines.append(("", text_color))
        elif stripped.startswith("#"):
            lines.append((stripped.replace("#", "").strip(), accent_color))
        else:
            wrapped = textwrap.wrap(paragraph, width=wrap_width) or [""]
            for w in wrapped:
                lines.append((w, text_color))

    fig, ax = plt.subplots(figsize=figsize, facecolor=bg_color)
    ax.set_facecolor(bg_color)
    ax.axis("off")

    line_height = 1.0
    text_objs = []
    for i, (line, color) in enumerate(lines):
        y = -i * line_height
        t = ax.text(
            0.03, y, line, va="top", ha="left",
            fontsize=fontsize if color == text_color else fontsize + 2,
            fontweight="normal" if color == text_color else "bold",
            color=color, transform=ax.transData,
        )
        text_objs.append(t)

    top = 0.5
    ax.set_xlim(0, 1)
    ax.set_ylim(top - lines_in_window, top)

    frames_per_line = max(1, int(fps * 0.12))
    total_scroll = max(1, len(lines) - lines_in_window)
    total_frames = total_scroll * frames_per_line

    def update(frame):
        shift = (frame / frames_per_line) * line_height
        new_top = top - shift
        ax.set_ylim(new_top - lines_in_window, new_top)
        return text_objs

    ani = animation.FuncAnimation(fig, update, frames=total_frames, blit=False)
    writer = animation.FFMpegWriter(fps=fps, bitrate=2500)

    logger.info(f"🎬 Rendering {total_frames} frames → {out_file}")
    ani.save(out_file, writer=writer)
    plt.close(fig)
    logger.info(f"✅ Video saved: {out_file}")


# Build combined text for video
if results:
    combined = "\n\n".join(
        f"# {r['filename']}\n\n{r['analysis']}" for r in results
    )
    if qa.history:
        combined += "\n\n# Questions & Answers\n\n"
        combined += "\n\n".join(
            f"## Q: {h['question']}\n\n{h['answer']}" for h in qa.history
        )

    video_path = str(Path(config.output_dir) / config.video_filename)
    create_scrolling_mp4(combined, out_file=video_path)
else:
    print("No results to render.")


In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 15 · Pipeline Summary
# ═══════════════════════════════════════════════════════

print("\n" + "═" * 55)
print("  🏭  PIPELINE EXECUTION SUMMARY")
print("═" * 55)
print(f"  Documents loaded:      {len(documents)}")
print(f"  Chunks created:        {len(chunks)}")
print(f"  Vectors in store:      {vectorstore.index.ntotal}")
print(f"  Analyses completed:    {len(results)}")
print(f"  Q&A questions answered: {len(qa.history)}")
print(f"  Model:                 {config.llm_model}")
print(f"  Embedding model:       {config.embedding_model}")
print(f"  Retrieval strategy:    {config.search_type} (k={config.retrieval_k})")
print(f"  Output directory:      {config.output_dir}/")
print("═" * 55)

total_time = sum(r.get("stats", {}).get("analysis_time_sec", 0) for r in results)
total_qa_time = sum(h.get("generation_time", 0) for h in qa.history)
print(f"  Total analysis time:   {total_time:.1f}s")
print(f"  Total Q&A time:        {total_qa_time:.1f}s")
print("═" * 55)
print("\n✅ All done. Your documents are analyzed and searchable!")
print("   → Re-run Cell 11 anytime for more Q&A.")
